In [76]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader 

In [77]:
# defining the transformations that will be done on the training data 
train_transforms =  transforms.Compose([
    transforms.Resize((64,64)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                        std=[0.229, 0.224, 0.225])
])

In [78]:
# loading the data 

train = datasets.ImageFolder(root=r'D:\Model Knowledge Base\CNN\Seha CNN project\images\train',transform=train_transforms)

In [79]:
train_loader = DataLoader(train, batch_size=32, shuffle=True, num_workers=4)

In [80]:
images, labels = next(iter(train_loader))

In [81]:
print(images.shape)
print(labels.shape)

torch.Size([32, 3, 64, 64])
torch.Size([32])


In [82]:
import torch    #main torch
import torch.nn as nn   #to create neural networks 
import torch.nn.functional as F   #to use functions like ReLU and SoftMax 

In [83]:
class CustomCNN(nn.Module):
    def __init__(self, num_classes=3):
        super(CustomCNN, self).__init__()
        
        self.conv1 = nn.Conv2d(in_channels=3, out_channels=16, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2,2)
        
        self.conv2 = nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, padding=1)
        
        self.fc1 = nn.Linear(32*16*16, 128)
        self.fc2 = nn.Linear(128, num_classes)
    
    def forward (self, x):
        #First Convolutional Layer 
        x = self.conv1(x)
        x = F.relu(x)
        x = self.pool(x)
        
        #Second Convolutional Layer 
        x = self.conv2(x)
        x = F.relu(x)
        x = self.pool(x)
        
        # Flattening 
        x = x.view(x.size(0), -1)

        #First Fully Connected Layer 
        x = self.fc1(x)
        x = F.relu(x)
        
        #Second Fully Connected Layer (OUTPUT)
        x = self.fc2(x)
        
        return x

## Training 

In [84]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = CustomCNN(num_classes=3).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=.001)

In [ ]:
num_epochs = 10

for epoch in range(num_epochs):
    model.train()  # مهم جدًا: يخلي الموديل في وضع التدريب
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        # 1️⃣ صفر الجريدينت قبل كل batch
        optimizer.zero_grad()

        # 2️⃣ Forward pass
        outputs = model(images)

        # 3️⃣ Loss
        loss = criterion(outputs, labels)

        # 4️⃣ Backward pass
        loss.backward()

        # 5️⃣ Update weights
        optimizer.step()

        # حساب loss وaccuracy
        running_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    acc = 100 * correct / total
    print(f"Epoch [{epoch+1}/{num_epochs}]  Loss: {running_loss:.4f}  Accuracy: {acc:.2f}%")


Epoch [1/10]  Loss: 17.6536  Accuracy: 49.38%
Epoch [2/10]  Loss: 8.6404  Accuracy: 78.75%
Epoch [3/10]  Loss: 4.7637  Accuracy: 85.42%
Epoch [4/10]  Loss: 3.1004  Accuracy: 92.92%
Epoch [5/10]  Loss: 1.7965  Accuracy: 97.08%
Epoch [6/10]  Loss: 1.0007  Accuracy: 99.17%
Epoch [7/10]  Loss: 0.4725  Accuracy: 99.58%
Epoch [8/10]  Loss: 0.2606  Accuracy: 100.00%
Epoch [9/10]  Loss: 0.1602  Accuracy: 99.79%
Epoch [10/10]  Loss: 0.0643  Accuracy: 100.00%


In [ ]:
# نفس transforms التدريب (resize + toTensor + normalize)
test_transforms = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406],
                        std=[0.229,0.224,0.225])
])

# test dataset
test_dataset = datasets.ImageFolder(root= r"D:\Model Knowledge Base\CNN\Seha CNN project\images\test", transform=test_transforms)

# test loader
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)


In [87]:
model.eval()  


CustomCNN(
  (conv1): Conv2d(3, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv2): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (fc1): Linear(in_features=8192, out_features=128, bias=True)
  (fc2): Linear(in_features=128, out_features=3, bias=True)
)

In [ ]:
correct = 0
total = 0

with torch.no_grad():  # معناه: مش هنعمل backward
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)
        
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)
        
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

test_acc = 100 * correct / total
print(f"Test Accuracy: {test_acc:.2f}%")


Test Accuracy: 90.83%
